# Notebook 02 — Preprocessing
Loads the Kaggle parquet dataset, applies the full preprocessing pipeline, and saves all splits to disk.

In [10]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [11]:
import subprocess, os
from pathlib import Path

REPO_URL  = 'https://github.com/calvinkatoroy/tugas-akhir-ai-kel-08.git'
REPO_NAME = 'tugas-akhir-ai-kel-08'

cwd = Path.cwd()

if (cwd / '../src').resolve().exists():
    result = subprocess.run(['git', '-C', str((cwd / '..').resolve()), 'pull'],
                            capture_output=True, text=True)
    print(result.stdout.strip() or 'Already up to date.')

elif (cwd / 'src').exists():
    result = subprocess.run(['git', 'pull'], capture_output=True, text=True)
    print(result.stdout.strip() or 'Already up to date.')
    os.chdir('notebooks')

else:
    repo_path = cwd / REPO_NAME
    if not repo_path.exists():
        print(f'Cloning {REPO_URL} ...')
        subprocess.run(['git', 'clone', REPO_URL], check=True)
    else:
        print('Repo found, pulling latest...')
        subprocess.run(['git', '-C', str(repo_path), 'pull'], capture_output=True)
    os.chdir(repo_path / 'notebooks')

print(f'Working dir: {Path.cwd()}')

Already up to date.
Working dir: /content/tugas-akhir-ai-kel-08/notebooks


In [12]:
import sys
sys.path.insert(0, '..')

import random
import numpy as np
import pandas as pd
import yaml

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

## Paths
Set `KAGGLE_DIR` (parquet source) and `SPLITS_DIR` (output).  
On Google Colab, mount Drive first and point both paths into Drive so splits persist across sessions.

In [13]:
import os
from pathlib import Path

IN_COLAB = 'google.colab' in str(get_ipython())

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive/tugas-akhir-ai')
    KAGGLE_DIR = DRIVE_ROOT / 'kaggle'
    SPLITS_DIR = DRIVE_ROOT / 'splits'
else:
    # Local run — paths relative to repo root
    with open('../config.yaml') as f:
        _cfg = yaml.safe_load(f)
    KAGGLE_DIR = Path('..') / _cfg['data']['kaggle_path']
    SPLITS_DIR = Path('..') / _cfg['data']['splits_path']

print(f'KAGGLE_DIR : {KAGGLE_DIR}')
print(f'SPLITS_DIR : {SPLITS_DIR}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
KAGGLE_DIR : /content/drive/MyDrive/tugas-akhir-ai/kaggle
SPLITS_DIR : /content/drive/MyDrive/tugas-akhir-ai/splits


## Run preprocessing pipeline

In [14]:
from src.preprocessing import run_preprocessing

info = run_preprocessing(
    config_path='../config.yaml',
    kaggle_dir=KAGGLE_DIR,
    splits_dir=SPLITS_DIR,
)
print('\nSummary:', info)

Loading parquets...
  Loaded DNS-testing.parquet: 6,703 rows
  Loaded LDAP-testing.parquet: 2,831 rows
  Loaded LDAP-training.parquet: 6,715 rows
  Loaded MSSQL-testing.parquet: 8,083 rows
  Loaded MSSQL-training.parquet: 10,974 rows
  Loaded NTP-testing.parquet: 134,674 rows
  Loaded NetBIOS-testing.parquet: 2,225 rows
  Loaded NetBIOS-training.parquet: 1,631 rows
  Loaded Portmap-training.parquet: 5,105 rows
  Loaded SNMP-testing.parquet: 4,018 rows
  Loaded Syn-testing.parquet: 907 rows
  Loaded Syn-training.parquet: 70,336 rows
  Loaded TFTP-testing.parquet: 121,833 rows
  Loaded UDP-testing.parquet: 12,462 rows
  Loaded UDP-training.parquet: 17,770 rows
  Loaded UDPLag-testing.parquet: 12,465 rows
  Loaded UDPLag-training.parquet: 12,639 rows
  Combined: 431,371 rows

Normalizing labels...
  Label distribution:
Label
ddos      333540
normal     97831
Name: count, dtype: int64

Selecting features...
  After feature selection + inf/nan drop: 431,371 rows, 16 features

Splitting 70/1

## Verify saved splits

In [15]:
X_train     = np.load(SPLITS_DIR / 'X_train.npy')
X_val       = np.load(SPLITS_DIR / 'X_val.npy')
X_test      = np.load(SPLITS_DIR / 'X_test.npy')
y_train     = np.load(SPLITS_DIR / 'y_train.npy')
y_val       = np.load(SPLITS_DIR / 'y_val.npy')
y_test      = np.load(SPLITS_DIR / 'y_test.npy')
X_train_seq = np.load(SPLITS_DIR / 'X_train_seq.npy')
X_val_seq   = np.load(SPLITS_DIR / 'X_val_seq.npy')
X_test_seq  = np.load(SPLITS_DIR / 'X_test_seq.npy')
y_train_seq = np.load(SPLITS_DIR / 'y_train_seq.npy')
y_val_seq   = np.load(SPLITS_DIR / 'y_val_seq.npy')
y_test_seq  = np.load(SPLITS_DIR / 'y_test_seq.npy')

print('=== Flat splits (for Random Forest) ===')
print(f'  X_train:  {X_train.shape}  y_train: {y_train.shape}')
print(f'  X_val:    {X_val.shape}    y_val:   {y_val.shape}')
print(f'  X_test:   {X_test.shape}   y_test:  {y_test.shape}')
print('\n=== Sequence splits (for LSTM/GRU) ===')
print(f'  X_train_seq: {X_train_seq.shape}  y_train_seq: {y_train_seq.shape}')
print(f'  X_val_seq:   {X_val_seq.shape}    y_val_seq:   {y_val_seq.shape}')
print(f'  X_test_seq:  {X_test_seq.shape}   y_test_seq:  {y_test_seq.shape}')

=== Flat splits (for Random Forest) ===
  X_train:  (301959, 16)  y_train: (301959,)
  X_val:    (64705, 16)    y_val:   (64705,)
  X_test:   (64707, 16)   y_test:  (64707,)

=== Sequence splits (for LSTM/GRU) ===
  X_train_seq: (301950, 10, 16)  y_train_seq: (301950,)
  X_val_seq:   (64696, 10, 16)    y_val_seq:   (64696,)
  X_test_seq:  (64698, 10, 16)   y_test_seq:  (64698,)


## Label balance across splits

In [16]:
for name, y in [('train', y_train), ('val', y_val), ('test', y_test)]:
    n_normal = (y == 0).sum()
    n_ddos   = (y == 1).sum()
    print(f'{name}: normal={n_normal:,} ({n_normal/len(y)*100:.1f}%)  '
          f'ddos={n_ddos:,} ({n_ddos/len(y)*100:.1f}%)')

train: normal=68,482 (22.7%)  ddos=233,477 (77.3%)
val: normal=14,674 (22.7%)  ddos=50,031 (77.3%)
test: normal=14,675 (22.7%)  ddos=50,032 (77.3%)


## Verify scaler (train mean ≈ 0, std ≈ 1)

In [17]:
import joblib

scaler = joblib.load(SPLITS_DIR / 'scaler.pkl')
print('Scaler mean (first 5 features):', scaler.mean_[:5].round(4))
print('Scaler std  (first 5 features):', scaler.scale_[:5].round(4))
print('X_train mean (should be ~0):',    X_train.mean(axis=0)[:5].round(4))
print('X_train std  (should be ~1):',    X_train.std(axis=0)[:5].round(4))

Scaler mean (first 5 features): [8.4253778e+06 2.3981300e+01 2.4067000e+00 9.4147658e+03 1.4971286e+03]
Scaler std  (first 5 features): [2.13010083e+07 1.65475900e+02 2.84389000e+01 3.40868123e+04
 5.60631443e+04]
X_train mean (should be ~0): [-0. -0.  0.  0.  0.]
X_train std  (should be ~1): [1.     0.9994 0.9987 0.9989 0.9997]


In [18]:
import pandas as pd
df_sample = pd.read_parquet('/content/drive/MyDrive/tugas-akhir-ai/kaggle/Syn-training.parquet')
print(df_sample.columns.tolist())


['Protocol', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Fwd Packets Length Total', 'Bwd Packets Length Total', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Packet Length Min', 'Packet Length Max', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count', 'SYN Flag Count', 'RST Flag Count', 'PSH Flag Count', 'ACK Flag Count', 'URG Flag Count', 'CWE Flag Count', 'ECE Flag Count', 'Down/Up Ra